# The super-awesome UC demo 101

## Setup SCIM at the account level

- [Microsoft Entra ID (Azure Active Directory)](https://learn.microsoft.com/en-us/azure/databricks/admin/users-groups/scim/aad)
- [Okta](https://docs.databricks.com/aws/en/admin/users-groups/scim/okta)
- [OneLogin](https://docs.databricks.com/aws/en/admin/users-groups/scim/onelogin)

## Setup Metastore and assign to a workspace

## Create storage credential
Azure:
1. Create Access Connector for Databricks
2. Assign the connector Storage Blob Data Contributor role on the storage account
3. Create Storage Credential in the UI with the Access Connector resource ID

AWS:



## Create external location


In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS gergeljkis_external_location
URL 'abfss://test@gergeljkissa.dfs.core.windows.net/project'
WITH (STORAGE CREDENTIAL `gergeljkis-storage-cred`);


## Create catalog

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS demo
MANAGED LOCATION 'abfss://test@gergeljkissa.dfs.core.windows.net/project';

## Disable legacy features (hive_metastore) in the workspace

Azure:
- [Disable access to the Hive metastore used by your Azure Databricks workspace](https://learn.microsoft.com/en-us/azure/databricks/data-governance/unity-catalog/disable-hms#disable-all-direct-access-to-the-hive-metastore)
- [Enforce user isolation cluster types on a workspace](https://learn.microsoft.com/en-us/azure/databricks/admin/workspace-settings/enforce-user-isolation)

AWS:
- [Disable access to the Hive metastore used by your Databricks workspace](https://docs.databricks.com/aws/en/data-governance/unity-catalog/disable-hms)
- [Enforce user isolation cluster types on a workspace](https://docs.databricks.com/aws/en/admin/workspace-settings/enforce-user-isolation)

In [0]:
%sql
USE CATALOG demo;

## Create schemas

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS demo.bronze;
CREATE SCHEMA IF NOT EXISTS silver;
CREATE SCHEMA IF NOT EXISTS demo.gold;

## Create sample managed and external tables in bronze, silver and gold

In [0]:
from pyspark.sql import Row

data = [Row(id=1, name="Alice"), Row(id=2, name="Bob"), Row(id=3, name="Charlie")]
df = spark.createDataFrame(data)

table_name = "demo.bronze.users"
df.write.format("delta").mode("overwrite").saveAsTable(table_name)

In [0]:
from pyspark.sql import Row

data = [Row(id=1, name="DEV"), Row(id=2, name="STG"), Row(id=3, name="PROD"), Row(id=4, name="POC")]
df = spark.createDataFrame(data)

table_name = "bronze.workspaces"
df.write.mode("overwrite").option("path", "abfss://test@gergeljkissa.dfs.core.windows.net/project/external/bronze/workspaces").saveAsTable(table_name)

In [0]:
from pyspark.sql import Row

data = [Row(user_id=1, workspace_id=1),
        Row(user_id=1, workspace_id=2),
        Row(user_id=1, workspace_id=3),
        Row(user_id=2, workspace_id=1),
        Row(user_id=2, workspace_id=2),
        Row(user_id=3, workspace_id=1)]
df = spark.createDataFrame(data)

table_name = "demo.bronze.workspace_assignments"
df.write.format("delta").mode("overwrite").saveAsTable(table_name)

In [0]:
%sql
CREATE OR REPLACE TABLE silver.users
LOCATION 'abfss://test@gergeljkissa.dfs.core.windows.net/project/external/silver/users'
AS SELECT * FROM bronze.users;

In [0]:
%sql
CREATE OR REPLACE TABLE silver.workspaces
AS SELECT * FROM bronze.workspaces;

In [0]:
%sql
CREATE OR REPLACE TABLE gold.workspace_access
AS SELECT w.name as workspace, u.name as username
FROM silver.workspaces w INNER JOIN bronze.workspace_assignments wa ON w.id = wa.workspace_id
INNER JOIN silver.users u ON wa.user_id = u.id;

In [0]:
%sql
CREATE OR REPLACE TABLE gold.workspace_statistics
AS SELECT w.name as workspace, count(u.id) as num_of_users
FROM silver.workspaces w LEFT OUTER JOIN bronze.workspace_assignments wa ON w.id = wa.workspace_id
LEFT OUTER JOIN silver.users u ON wa.user_id = u.id
GROUP BY w.id, w.name;

## Create volumes

In [0]:
%sql
CREATE SCHEMA demo.landing;

In [0]:
%sql
CREATE EXTERNAL VOLUME demo.landing.data
LOCATION 'abfss://test@gergeljkissa.dfs.core.windows.net/project/data'

In [0]:
df = spark.read.option("header", "true").csv("/Volumes/demo/landing/data/csv1.csv")
display(df)

In [0]:
df = spark.table('demo.bronze.users')
df.write.mode("overwrite").format('csv').save('/Volumes/demo/landing/data/demo.csv')